In [10]:
from dotenv import load_dotenv

load_dotenv()

True

In [11]:
from openai import OpenAI

openai_client = OpenAI()

In [ ]:
# _____ Question_01 - How many lesson pages ______

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
  repo_owner = "DataTalksClub",
  repo_name = "llm-zoomcamp",
  commit_id = "8c1834d",
  allowed_extensions = {"md"},
  filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
  doc = file.parse()
  documents.append(doc)

In [ ]:
len(documents)

72

In [ ]:
# _____ Question__2. Indexing and searching ______

In [7]:
from minsearch import Index

index = Index(
  text_fields = ["content"],
  keyword_fields=["filename"],
)

index.fit(documents)

In [20]:
question = 'How does the agentic loop keep calling the model until it stops?'

In [ ]:
search_results = index.search(
  question,
  num_results = 5
)

search_results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [ ]:
# _____ Question_03 - How many input (prompt) tokens did we send to the model for this request? ______

In [ ]:
# Обновленный raq из курса
from hw_rag_helper import RAGBase

rag_base = RAGBase(
  index = index, 
  llm_client = openai_client, 
  model = "gpt-5.4-mini"
)

In [ ]:
answer = rag_base.rag(question)
answer

'It keeps calling the model in a `while True` loop.\n\nAfter each model response, the code checks whether the response contains any `function_call` items. If it does, it runs the tool, appends the tool result to `messages`, and loops again. If there are no function calls in that turn, it breaks out of the loop.\n\nSo the stop condition is:\n\n- **function calls found** → keep looping\n- **no function calls** → stop and return the final answer'

In [ ]:
response = rag_base.get_llm_response()
usage = response.usage
input = usage.input_tokens

input

7052

In [ ]:
# _____ Question_04 - How many chunks do you get? ______

In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size = 2000, step = 1000)
len(chunks)

295

In [ ]:
# _____ Question_05 - How many chunks do you get? ______

In [14]:
from minsearch import Index

index = Index(
  text_fields = ["content"],
  keyword_fields=["filename"],
)

In [15]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size = 2000, step = 1000)
index.fit(chunks)

In [ ]:
from hw_rag_helper import RAGBase

rag_base = RAGBase(
  index = index, 
  llm_client = openai_client, 
  model = "gpt-5.4-mini"
)

In [21]:
answer = rag_base.rag(question)
answer

'The loop keeps calling the model inside a `while True` loop and stops when a turn produces no `function_call` items.\n\nSpecifically:\n- It sets `has_function_calls = False` at the start of each iteration.\n- It calls the model and processes `response.output`.\n- If any item is a `function_call`, it runs the tool, appends the result, and sets `has_function_calls = True`.\n- After processing everything, it checks `if has_function_calls == False:` and then `break`s.\n\nSo the model controls how many tool calls happen, and the loop ends only when the model returns a final message with no more tool calls.'

In [22]:
response = rag_base.get_llm_response()
usage = response.usage
input = usage.input_tokens

input

2237

In [ ]:
# _____ Question_06 - Turning it into an agent and how many times did the agent call search? ______

In [23]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [ ]:
instructions = '''
  You're a course teaching assistant. Answer the student's question 
  using the search tool. Make multiple searches with different keywords 
  before answering.
'''.strip()

question = 'How does the agentic loop work, and how is it different from plain RAG?'

messages = [
  {'role': 'developer', 'content': instructions},
  {'role': 'user', 'content': question}
]

In [ ]:
def search(query: str) -> dict[str, str]:

    return index.search(
        query,
        num_results=5,
    )

In [26]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [27]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'No description provided.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [28]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
  tools = agent_tools,
  developer_prompt = instructions,
  chat_interface = chat_interface,
  llm_client = OpenAIClient(model="gpt-5.4-mini")
)

In [ ]:
result = runner.loop(
  prompt = question,
  callback = callback,
)